<a href="https://colab.research.google.com/github/jazibrana7-ctrl/Reg-tech-app/blob/main/Home_Budget_Tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏠 Home Budget Tracker
## Pakistani Household Expense Manager

**Created for:** Jazib Ali  
**Purpose:** Track all home expenses, handle fixed/variable costs, and split bills between family members.

### Features:
✅ Auto-fetch petrol prices from web  
✅ Lock expenses that don't change monthly  
✅ Calculate fuel costs by vehicle (liters × price)  
✅ Track meat/milk by quantity  
✅ Add custom expense categories anytime  
✅ Split total between you and father  
✅ **All data saved to Google Drive** (survives logout)  

---


In [1]:
import json, os
from datetime import datetime

try:
    import ipywidgets as widgets
except ImportError:
    get_ipython().system('pip install -q ipywidgets')
    import ipywidgets as widgets

from IPython.display import display, HTML, clear_output

# ---- Connect to Google Drive so your data is remembered next time you log in ----
DATA_DIR = "/content"
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DATA_DIR = "/content/drive/MyDrive/HomeBudgetApp"
    os.makedirs(DATA_DIR, exist_ok=True)
    print("✅ Google Drive connected — your numbers will be saved permanently.")
except Exception:
    print("⚠️ Drive not connected, data will reset once this session ends.")
    os.makedirs(DATA_DIR, exist_ok=True)

DATA_FILE = os.path.join(DATA_DIR, "budget_data.json")

DEFAULT_DATA = {
    "values": {},
    "fixed_flags": {},
    "transport": {"fuel_price": 0, "civic_liters": 0, "cultus_liters": 0, "bike_liters": 0},
    "meat": {"price": 0, "kg": 0},
    "milk": {"price": 0, "liters_per_day": 0, "days": 30},
    "custom_expenses": {},
    "last_jazib_share": 0,
    "last_updated": None,
}

def load_data():
    if os.path.exists(DATA_FILE):
        try:
            with open(DATA_FILE, "r") as f:
                saved = json.load(f)
            data = json.loads(json.dumps(DEFAULT_DATA))  # deep copy of defaults
            data.update(saved)
            return data
        except json.JSONDecodeError:
            pass
    return json.loads(json.dumps(DEFAULT_DATA))

def save_data(data):
    data["last_updated"] = datetime.now().strftime("%d %b %Y, %I:%M %p")
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=2)
    print(f"💾 Saved to {DATA_FILE}")

DATA = load_data()



Mounted at /content/drive
✅ Google Drive connected — your numbers will be saved permanently.


In [2]:
display(HTML("""
<style>
.budget-header{
  background: linear-gradient(135deg,#0f766e 0%,#134e4a 100%);
  color:white; padding:22px 26px; border-radius:16px; margin-bottom:18px;
  font-family:'Segoe UI', Arial, sans-serif;
}
.budget-header h1{margin:0; font-size:25px;}
.budget-header p{margin:5px 0 0; opacity:.85; font-size:13.5px;}
.section-title{
  font-family:'Segoe UI', Arial, sans-serif; font-size:16px; font-weight:600;
  color:#134e4a; margin:4px 0 10px 0;
}
.row-label{
  width:230px; font-family:'Segoe UI', Arial, sans-serif; font-size:14px; color:#374151;
  padding-top:4px;
}
.stat-box{
  display:inline-block; min-width:175px; padding:14px 18px; border-radius:12px; margin:6px 10px 6px 0;
  font-family:'Segoe UI', Arial, sans-serif; text-align:center; vertical-align:top;
}
.stat-total{background:#ecfdf5; border:1px solid #6ee7b7;}
.stat-jazib{background:#eff6ff; border:1px solid #93c5fd;}
.stat-father{background:#fff7ed; border:1px solid #fdba74;}
.stat-box .lbl{font-size:12.5px; color:#555;}
.stat-box .val{font-size:21px; font-weight:700; margin-top:4px; color:#111;}
.summary-table{width:100%; border-collapse:collapse; font-family:'Segoe UI', Arial, sans-serif; font-size:14px; margin-top:6px;}
.summary-table td{padding:6px 4px; border-bottom:1px solid #eee;}
.summary-table td:last-child{text-align:right; font-weight:600;}
.last-saved{color:#888; font-size:12px; font-family:'Segoe UI', Arial, sans-serif; margin-top:8px;}
</style>
"""))

display(HTML("""
<div class="budget-header">
  <h1>🏠 Home Budget Tracker</h1>
  <p>Type your monthly numbers below, lock the ones that don't change, and hit Calculate.</p>
</div>
"""))



In [3]:
import re

def fetch_petrol_price():
    """
    Best-effort attempt to grab today's petrol price from a public page.
    There is no free official government API for fuel prices in Pakistan,
    so this just scrapes a public page and can break if that page changes.
    Manual entry (the box next to this button) is always the reliable option.
    """
    try:
        import requests
        r = requests.get(
            "https://www.pakwheels.com/petroleum-prices-in-pakistan",
            timeout=8,
            headers={"User-Agent": "Mozilla/5.0"},
        )
        text = r.text
        m = re.search(r'Petrol Price in Pakistan is\s*(?:PKR|Rs\.?)\s*([\d]+\.?\d*)', text, re.IGNORECASE)
        if not m:
            m = re.search(r'Rs\.?\s*([\d]{2,3}\.\d{1,2})\s*/?\s*Ltr', text, re.IGNORECASE)
        if m:
            return float(m.group(1))
    except Exception:
        pass
    return None



In [4]:
# ---------------- Expense schema ----------------
# (category, key, label, icon)
SIMPLE_ITEMS = [
    ("family", "waliha_pocket",  "Waliha Pocket Money", "💵"),
    ("family", "waliha_tuition", "Waliha Tuition Fees",  "📚"),
    ("family", "waliha_school",  "Waliha School Fees",   "🏫"),
    ("family", "rahim_pocket",   "Rahim Pocket Money",   "💵"),
    ("family", "jazib_pocket",   "Jazib Pocket Money",   "💵"),
    ("family", "jazib_academy",  "Jazib Academy Fees",   "📖"),
    ("family", "mother_pocket",  "Mother Pocket Money",  "💵"),

    ("food",   "veg_fruit",      "Vegetables & Fruits",  "🥦"),
    ("food",   "grocery",        "Grocery",               "🛒"),

    ("utilities", "electricity", "Electricity Bill",      "💡"),
    ("utilities", "water",       "Water Bill",             "🚰"),
    ("utilities", "sui_gas",     "Sui Gas Bill",           "🔥"),
    ("utilities", "internet",    "Internet Fees",          "🌐"),
    ("utilities", "trash",       "Trash Fees",             "🗑️"),
    ("utilities", "cable",       "Cable Fees",             "📺"),
    ("utilities", "newspaper",   "Newspaper Bill",         "📰"),

    ("staff",  "maid",           "Maid Fees",              "🧹"),   # assumed "Mate fees" = Maid fees, rename below if not
    ("staff",  "gardener",       "Gardener Fees",          "🌳"),
    ("staff",  "guard",          "Guard Fees",             "🛡️"),
    ("staff",  "car_wash",       "Car Wash",               "🚿"),
]

CATEGORY_TITLES = {
    "family":     "👨‍👩‍👧‍👦 Family Pocket Money & Fees",
    "food":       "🍎 Food & Kitchen",
    "utilities":  "💡 Utilities & Bills",
    "staff":      "🧑‍🔧 Staff & Misc",
}

item_widgets = {}  # key -> (amount_widget, fixed_checkbox_widget)

def make_item_row(key, label, icon):
    val = DATA["values"].get(key, 0)
    fixed = DATA["fixed_flags"].get(key, False)

    label_w = widgets.HTML(f"<div class='row-label'>{icon} {label}</div>")
    amount_w = widgets.BoundedFloatText(
        value=val, min=0, max=10_000_000, step=50,
        layout=widgets.Layout(width="150px"),
    )
    fixed_w = widgets.Checkbox(
        value=fixed, description="Fixed 🔒", indent=False,
        layout=widgets.Layout(width="110px"),
    )
    amount_w.disabled = fixed

    def on_fixed_change(change):
        amount_w.disabled = change["new"]
    fixed_w.observe(on_fixed_change, names="value")

    item_widgets[key] = (amount_w, fixed_w)
    return widgets.HBox([label_w, amount_w, fixed_w])

def build_section(category):
    rows = [make_item_row(key, label, icon)
            for (cat, key, label, icon) in SIMPLE_ITEMS if cat == category]
    return widgets.VBox(
        [widgets.HTML(f"<div class='section-title'>{CATEGORY_TITLES[category]}</div>")] + rows
    )

family_section    = build_section("family")
food_basic_section = build_section("food")
utilities_section = build_section("utilities")
staff_section      = build_section("staff")



In [5]:
# ---- Transport Section ----
transport_fuel_price = widgets.FloatText(
    value=DATA["transport"]["fuel_price"],
    placeholder="e.g. 350",
    layout=widgets.Layout(width="120px"),
    description="Fuel Price (Rs/L):"
)

auto_fetch_btn = widgets.Button(
    description="🔄 Auto-Fetch Price",
    button_style="info",
    layout=widgets.Layout(width="150px"),
)

def fetch_and_update(btn):
    with auto_fetch_output:
        auto_fetch_output.clear_output()
        print("⏳ Fetching petrol price from web...")
        price = fetch_petrol_price()
        if price:
            transport_fuel_price.value = price
            print(f"✅ Updated to Rs. {price:.2f}/L (latest rate)")
        else:
            print("❌ Could not fetch price. Please enter manually.")

auto_fetch_btn.on_click(fetch_and_update)
auto_fetch_output = widgets.Output()

civic_liters = widgets.FloatText(
    value=DATA["transport"]["civic_liters"],
    placeholder="e.g. 80",
    layout=widgets.Layout(width="120px"),
    description="Civic (Liters):",
)
cultus_liters = widgets.FloatText(
    value=DATA["transport"]["cultus_liters"],
    placeholder="e.g. 50",
    layout=widgets.Layout(width="120px"),
    description="Cultus (Liters):",
)
bike_liters = widgets.FloatText(
    value=DATA["transport"]["bike_liters"],
    placeholder="e.g. 30",
    layout=widgets.Layout(width="120px"),
    description="Bike (Liters):",
)

display(widgets.HTML(
    "<div class='section-title'>🚗 Transport (Fuel Consumption)</div>"
))
display(widgets.HBox([transport_fuel_price, auto_fetch_btn]))
display(auto_fetch_output)
display(widgets.HTML("<p style='font-size:12px; color:#666; margin:4px 0;'>Enter liters consumed per month for each vehicle:</p>"))
display(widgets.HBox([civic_liters, cultus_liters, bike_liters]))



HTML(value="<div class='section-title'>🚗 Transport (Fuel Consumption)</div>")

Output()

HTML(value="<p style='font-size:12px; color:#666; margin:4px 0;'>Enter liters consumed per month for each vehi…

In [6]:
# ---- Meat & Milk Section ----
meat_price = widgets.FloatText(
    value=DATA["meat"]["price"],
    placeholder="e.g. 1200",
    layout=widgets.Layout(width="120px"),
    description="Meat Price (Rs/kg):",
)
meat_kg = widgets.FloatText(
    value=DATA["meat"]["kg"],
    placeholder="e.g. 5",
    layout=widgets.Layout(width="120px"),
    description="Meat Quantity (kg/month):",
)

milk_price = widgets.FloatText(
    value=DATA["milk"]["price"],
    placeholder="e.g. 150",
    layout=widgets.Layout(width="120px"),
    description="Milk Price (Rs/liter):",
)
milk_liters_per_day = widgets.FloatText(
    value=DATA["milk"]["liters_per_day"],
    placeholder="e.g. 1.5",
    layout=widgets.Layout(width="120px"),
    description="Milk (liters/day):",
)
milk_days = widgets.IntSlider(
    value=int(DATA["milk"]["days"]),
    min=1, max=31, step=1,
    description="Days/month:",
    layout=widgets.Layout(width="220px"),
)

display(widgets.HTML(
    "<div class='section-title'>🥩 Meat & Milk</div>"
))
display(widgets.HBox([meat_price, meat_kg]))
display(widgets.HBox([milk_price, milk_liters_per_day, milk_days]))



HTML(value="<div class='section-title'>🥩 Meat & Milk</div>")

## 📌 Family Pocket Money & School Fees


In [7]:
display(family_section)


## 🍎 Food & Kitchen Costs


In [8]:
display(food_basic_section)


## 💡 Utilities & Bills


In [9]:
display(utilities_section)


## 🧑‍🔧 Staff & Miscellaneous


In [10]:
display(staff_section)


In [11]:
# ---- Custom Expenses ----
custom_widgets = {}

def render_custom_expenses():
    global custom_widgets
    custom_widgets.clear()

    custom_list = []
    if DATA["custom_expenses"]:
        for custom_key, (custom_name, custom_val) in DATA["custom_expenses"].items():
            val_w = widgets.FloatText(
                value=custom_val, min=0, max=10_000_000, step=50,
                layout=widgets.Layout(width="150px"),
            )
            custom_widgets[custom_key] = val_w
            label_w = widgets.HTML(f"<div class='row-label'>🏷️ {custom_name}</div>")
            custom_list.append(widgets.HBox([label_w, val_w]))

    custom_display.clear_output(wait=True)
    with custom_display:
        if custom_list:
            display(widgets.HTML("<div class='section-title'>➕ Custom Expenses (Your Own)</div>"))
            display(widgets.VBox(custom_list))

custom_display = widgets.Output()
render_custom_expenses()
display(custom_display)

# Add custom expense button
add_custom_name = widgets.Text(
    placeholder="e.g. 'Pet Food' or 'Charity'",
    layout=widgets.Layout(width="200px"),
    description="New Expense:"
)
add_custom_amount = widgets.FloatText(
    value=0, min=0, max=10_000_000, step=50,
    layout=widgets.Layout(width="120px"),
    description="Amount (Rs):"
)
add_custom_btn = widgets.Button(description="➕ Add", button_style="success", layout=widgets.Layout(width="80px"))

def on_add_custom(btn):
    name = add_custom_name.value.strip()
    if not name:
        print("⚠️ Please enter a name for the new expense.")
        return
    custom_key = f"custom_{len(DATA['custom_expenses']) + 1}"
    DATA["custom_expenses"][custom_key] = (name, add_custom_amount.value)
    add_custom_name.value = ""
    add_custom_amount.value = 0
    render_custom_expenses()
    print(f"✅ Added '{name}'")

add_custom_btn.on_click(on_add_custom)
display(widgets.HBox([add_custom_name, add_custom_amount, add_custom_btn]))



Output()

In [12]:
# ---- Deduction & Calculation ----
jazib_share_input = widgets.FloatText(
    value=DATA.get("last_jazib_share", 0),
    min=0, max=10_000_000, step=100,
    layout=widgets.Layout(width="150px"),
    description="Your Share (Rs):"
)

display(widgets.HTML(
    "<div class='section-title'>📊 Settlement (Optional)</div>"
))
display(widgets.HTML(
    "<p style='font-size:13px; color:#555;'>If you pay some expenses but your father covers the rest, enter what you paid. We'll calculate his share.</p>"
))
display(jazib_share_input)

# ---- CALCULATE BUTTON ----
calc_btn = widgets.Button(
    description="🔢 CALCULATE",
    button_style="warning",
    layout=widgets.Layout(width="200px", height="40px"),
    tooltip="Compute total, your share, and your father's share"
)

output_area = widgets.Output()

def on_calculate(btn):
    output_area.clear_output(wait=True)
    with output_area:
        # Gather all values
        DATA["values"].clear()
        DATA["fixed_flags"].clear()

        for key, (amount_w, fixed_w) in item_widgets.items():
            DATA["values"][key] = amount_w.value
            DATA["fixed_flags"][key] = fixed_w.value

        # Transport
        DATA["transport"]["fuel_price"] = transport_fuel_price.value
        DATA["transport"]["civic_liters"] = civic_liters.value
        DATA["transport"]["cultus_liters"] = cultus_liters.value
        DATA["transport"]["bike_liters"] = bike_liters.value

        # Meat & Milk
        DATA["meat"]["price"] = meat_price.value
        DATA["meat"]["kg"] = meat_kg.value
        DATA["milk"]["price"] = milk_price.value
        DATA["milk"]["liters_per_day"] = milk_liters_per_day.value
        DATA["milk"]["days"] = milk_days.value

        # Custom
        for custom_key, val_w in custom_widgets.items():
            if custom_key in DATA["custom_expenses"]:
                name, _ = DATA["custom_expenses"][custom_key]
                DATA["custom_expenses"][custom_key] = (name, val_w.value)

        DATA["last_jazib_share"] = jazib_share_input.value

        # ===== CALCULATE TOTAL =====
        total = sum(DATA["values"].values())

        # Transport
        transport_total = (
            DATA["transport"]["fuel_price"] *
            (DATA["transport"]["civic_liters"] +
             DATA["transport"]["cultus_liters"] +
             DATA["transport"]["bike_liters"])
        )
        total += transport_total

        # Meat
        meat_total = DATA["meat"]["price"] * DATA["meat"]["kg"]
        total += meat_total

        # Milk
        milk_total = (DATA["milk"]["price"] *
                      DATA["milk"]["liters_per_day"] *
                      DATA["milk"]["days"])
        total += milk_total

        # Custom
        custom_total = sum(v for _, v in DATA["custom_expenses"].values())
        total += custom_total

        # Apply deduction
        jazib_share = jazib_share_input.value
        father_share = max(0, total - jazib_share)

        # Save to Drive
        save_data(DATA)

        # ===== DISPLAY RESULTS =====
        display(HTML(f"""
        <div style="margin-top:20px;">
            <div class="stat-box stat-total">
                <div class="lbl">💰 TOTAL MONTHLY EXPENSE</div>
                <div class="val">Rs. {total:,.0f}</div>
            </div>
            <div class="stat-box stat-jazib">
                <div class="lbl">👤 Your Share (Jazib)</div>
                <div class="val">Rs. {jazib_share:,.0f}</div>
            </div>
            <div class="stat-box stat-father">
                <div class="lbl">👨 Father's Share</div>
                <div class="val">Rs. {father_share:,.0f}</div>
            </div>
        </div>
        """))

        # Breakdown table
        display(HTML("<h3 style='font-family:\"Segoe UI\", Arial; font-size:15px; color:#134e4a; margin:16px 0 8px 0;'>📋 Breakdown</h3>"))

        breakdown = []
        for (cat, key, label, icon) in SIMPLE_ITEMS:
            val = DATA["values"].get(key, 0)
            if val > 0:
                breakdown.append((f"{icon} {label}", val))

        if transport_total > 0:
            breakdown.append(("🚗 Transport (Fuel)", transport_total))
        if meat_total > 0:
            breakdown.append(("🥩 Meat", meat_total))
        if milk_total > 0:
            breakdown.append(("🥛 Milk", milk_total))
        for custom_key, (name, val) in DATA["custom_expenses"].items():
            if val > 0:
                breakdown.append((f"🏷️ {name}", val))

        breakdown.sort(key=lambda x: x[1], reverse=True)

        table_html = "<table class='summary-table'><tr><td style='text-align:left;'><b>Item</b></td><td><b>Amount</b></td></tr>"
        for label, amt in breakdown:
            table_html += f"<tr><td style='text-align:left;'>{label}</td><td>Rs. {amt:,.0f}</td></tr>"
        table_html += "</table>"

        display(HTML(table_html))

        if DATA.get("last_updated"):
            display(HTML(f"<div class='last-saved'>Last saved: {DATA['last_updated']}</div>"))

calc_btn.on_click(on_calculate)

display(calc_btn)
display(output_area)



HTML(value="<div class='section-title'>📊 Settlement (Optional)</div>")

HTML(value="<p style='font-size:13px; color:#555;'>If you pay some expenses but your father covers the rest, e…

FloatText(value=0.0, description='Your Share (Rs):', layout=Layout(width='150px'), step=100.0)

Button(button_style='warning', description='🔢 CALCULATE', layout=Layout(height='40px', width='200px'), style=B…

Output()